# Clean, Source-Organized Copy
Saved outputs and machine-specific metadata are intentionally excluded. The byte-identical original remains in the local audit archive. Prepare the local workbench before running this notebook. Notebooks 01 and 02 execute GPU inference when explicitly enabled; 03 disables its optional GPU activation check by default.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "scripts" / "lab.py").is_file()
)
WORKSPACE = PROJECT_ROOT / ".local" / "workbench"
if not (WORKSPACE / "weight_edit.py").is_file():
    raise RuntimeError("Run python scripts/lab.py prepare --with-local-data first.")
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))


# Accuracy vs. token budget: unedited Qwen3.5-0.8B

Read the saved, separately declared GSM8K TRAIN development experiment. This notebook does not load a model or launch inference. All statistical tables are recomputed from raw records, not copied from summary.json.

GSM8K TRAIN is likely in Qwen's training data; per-item inclusion is unverified. Absolute accuracy here is not a capability or benchmark claim. Select **Python (weight-space)** and run the cells in order.

In [ ]:
import json
from pathlib import Path

from IPython.display import Image, Markdown, display

from budget_sweep_outputs import diagnostics_markdown, fenced, load_saved_run, paired_markdown, tables_markdown

RUN_EXPERIMENT = False
if RUN_EXPERIMENT:
    raise RuntimeError('Inference is deliberately CLI-only to prevent repeating the fixed run. Review the approved worker and use its resume command explicitly.')

RUN_DIR = Path.cwd() / 'results' / 'budget_sweep_20261106T191821_788114Z'
plan, pool, records, saved, derived, budget = load_saved_run(RUN_DIR)
print('Saved run status:', saved['status'])
print('Main responses:', derived['main_responses'], '/ 60; total including validation:', len(records))
print('Complete 30-question comparison:', derived['complete_data'])
print('Budget accounting:', json.dumps(budget, indent=2))
print('EOS remains 248044 only; 248046 is a diagnostic, never an added stopping token.')

## Recomputed cap tables

Correct at cap k requires natural EOS at or before k and the frozen strict boxed-integer grade. Each cap comes from the same single long trajectory. Wilson intervals apply to accuracy and natural-completion proportions, not mean tokens.

In [ ]:
display(Markdown(tables_markdown(derived)))
display(Markdown('## Paired Thinking-On vs. Thinking-Off\n\n' + paired_markdown(derived)))
print('Predeclared descriptive pattern:', json.dumps(derived['hypothesis_pattern'], indent=2))
print('With n=30 only large effects (~25+ points) are detectable; non-significance is not equivalence.')

In [ ]:
figure = RUN_DIR / 'accuracy_vs_token_budget.png'
if figure.is_file():
    display(Image(filename=str(figure)))
else:
    print('The saved PNG is not available yet; statistical tables are still recomputed above.')

## Truncation, diagnostics and runtime

Prefix equality compares integer token IDs, including early EOS. Main and short-validation token-248046 counts are kept separate. The ETA is a measurement-based projection, never a wall-time stop.

In [ ]:
print('Prefix checks:', json.dumps(derived['prefix_validation'], indent=2))
display(Markdown(diagnostics_markdown(derived)))
print('Observed deterministic settings:', json.dumps(saved.get('runtime', {}), indent=2))
print('Post-truncation profile:', json.dumps(saved.get('truncation_milestone'), indent=2))
print('Original target hashes:', json.dumps(saved['target_hash_checks'], indent=2))

In [ ]:
print('Naturally completed wrong main responses <=1024 tokens:', len(derived['short_wrong_answers']))
for item in derived['short_wrong_answers']:
    print(item['problem_id'], item['condition'], 'seed', item['seed'], 'tokens', item['tokens'], 'parsed', item['final_answer'], 'gold', item['expected'])
    display(Markdown(fenced(item['text'], limit=6000)))

## Stop

This is a fixed development-set investigation, not evidence that weights changed or that the model learned a new capability. Instructions are held fixed, so this run cannot isolate the token cap rather than instruction as the primary cause of the prior result. No further generations or experiments are scheduled by this notebook.